In [1]:
# !pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 34.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 28.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 55.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 55.4 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 56.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 36.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 53.0 MB/s  0:00:016m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 50.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 30.2 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 40.0 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━

In [3]:
import os
import csv
import hashlib
from pathlib import Path
from PIL import Image, ExifTags
from ultralytics import YOLO

class ImageAnnotator:
    def __init__(self, model_name='yolov8n.pt'):
        """Initialize with a fast pre-trained YOLO model."""
        self.model = YOLO(model_name)
        self.annotations = []
        self.metadata = []

    def get_image_metadata(self, img_path):
        """Extract basic metadata from the image file."""
        img = Image.open(img_path)
        info = img.info
        stat = os.stat(img_path)
        meta = {
            'filename': img_path.name,
            'size_bytes': stat.st_size,
            'format': img.format,
            'width': img.width,
            'height': img.height
        }
        return meta

    def process_folder(self, folder_path):
        """Read all images and run detection."""
        path = Path(folder_path)
        image_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')
        image_files = [f for f in path.iterdir() if f.suffix.lower() in image_extensions]

        serial_no = 1
        for img_path in image_files:
            # 1. Store Metadata
            self.metadata.append(self.get_image_metadata(img_path))

            # 2. Run Object Detection
            results = self.model(str(img_path), verbose=False)
            
            for result in results:
                # Get detected classes
                classes = result.names
                found_boxes = result.boxes

                if len(found_boxes) == 0:
                    # Handle image with no objects detected
                    self.annotations.append({
                        'Serial Number': serial_no,
                        'Image Name': img_path.name,
                        'Found Object': 'None',
                        'Group ID': 'Empty'
                    })
                    serial_no += 1
                else:
                    # One row per object found
                    for box in found_boxes:
                        obj_name = classes[int(box.cls[0])]
                        # Group ID is generated by hashing the object name to group similar images
                        group_id = hashlib.md5(obj_name.encode()).hexdigest()[:8]
                        
                        self.annotations.append({
                            'Serial Number': serial_no,
                            'Image Name': img_path.name,
                            'Found Object': obj_name,
                            'Group ID': group_id
                        })
                        serial_no += 1

    def save_to_csv(self, output_dir):
        """Save results to CSV files in the specified directory."""
        # Save Annotations
        annot_file = Path(output_dir) / 'image_annotations.csv'
        with open(annot_file, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['Serial Number', 'Image Name', 'Found Object', 'Group ID'])
            writer.writeheader()
            writer.writerows(self.annotations)

        # Save Metadata
        meta_file = Path(output_dir) / 'image_metadata.csv'
        with open(meta_file, 'w', newline='') as f:
            if self.metadata:
                writer = csv.DictWriter(f, fieldnames=self.metadata[0].keys())
                writer.writeheader()
                writer.writerows(self.metadata)

        print(f"Successfully saved:\n1. {annot_file}\n2. {meta_file}")

# --- Execution ---
if __name__ == "__main__":
    # Setup folder (defaults to current directory or create a sample)
    current_dir = os.getcwd()
    
    # Initialize and Run
    annotator = ImageAnnotator()
    
    # Note: Ensure you have images in the directory or specify a path
    print("Processing images...")
    annotator.process_folder(current_dir + '/image')
    annotator.save_to_csv(current_dir + '/Output')

ImportError: libGL.so.1: cannot open shared object file: No such file or directory